## This is Chain based 

In [2]:
!pip install langchain langchain-community langchain-core


In [11]:
# pip show langchain
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-openai
!pip install -U langchain-google-genai
!pip install -U sqlalchemy

  Attempting uninstall: langchain-google-genai
    Found existing installation: langchain-google-genai 4.3.1
    Uninstalling langchain-google-genai-4.3.1:
      Successfully uninstalled langchain-google-genai-4.3.1
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 1.1 MB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.2 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.39
    Uninstalling SQLAlchemy-2.0.39:
      Successfully uninstalled SQLAlchemy-2.0.39


  You can safely remove it manually.


In [15]:
# Impport necessary libraries
from langchain_community.utilities import SQLDatabase
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate

# from langchain.chat_models import ChatOpenAI
# from langchain.chains import create_sql_query_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI


In [17]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

if not GOOGLE_API_KEY:
    print("❌ **GOOGLE_API_KEY missing**. Add it to `.streamlit/secrets.toml` or environment variables.")


print("🔄 Initializing Gemini Flash...")
def get_llm():
    return ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        google_api_key=GOOGLE_API_KEY,
        temperature=0.2,
        max_tokens=2000,
    )

llm = get_llm()


🔄 Initializing Gemini Flash...


In [18]:
# Connect your MySQL database
# Make sure to install the required packages
host = 'localhost'
port = '3306'
username = 'root'
password = 'root'
database_schema = 'uber'
mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=2)

In [19]:
# Database connection
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=1)

In [22]:
import sqlalchemy
import langchain
import langchain_community

print("SQLAlchemy:", sqlalchemy.__version__)
print("LangChain:", langchain.__version__)
print("LangChain Community:", langchain_community.__version__)


SQLAlchemy: 2.0.39
LangChain: 1.3.9
LangChain Community: 0.4.2


In [21]:
db.get_table_info()

AttributeError: 'PrimaryKeyConstraint' object has no attribute 'get_dialect_option'

In [5]:
# Create the LLM Prompt Template                  
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide me the sql query dont include anything else.
           Provide me sql query in a single line dont add line breaks.
Table Schema:
{schema}

Question: {question}
SQL Query:
"""
prompt = ChatPromptTemplate.from_template(template)

In [6]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema


In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    api_key="API_KEY_HERE"
)

In [8]:
# Create the SQL query chain using the LLM and the prompt template
sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: get_schema(db))
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [16]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question": "What was the budget of Product 12"})
print(resp)

```sql
SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'
```


In [26]:
import re

query = re.search(r"```sql\s*(.*?)\s*```", resp, re.DOTALL | re.IGNORECASE)

if query:
    query=query.group(1).strip()


In [27]:
db.run(query)

'[(1356976.996,)]'